# Task 4: Statistical Modeling & Risk-Based Pricing

## Objective
Build and evaluate predictive models that form the core of a dynamic, risk-based pricing system.

### Modeling Goals:
1. **Claim Severity Prediction (Risk Model)**: Predict `TotalClaims` for policies with claims.
2. **Premium Optimization (Pricing Framework)**: Predict an appropriate premium using a combined approach:
   `Premium = (P(claim) × Predicted Severity) + Expense Loading + Profit Margin`

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add src to path
sys.path.append(os.path.abspath('../src'))

from data_loader import load_raw_data, preprocess
from modeling import (
    drop_high_missing, engineer_features, encode_categoricals,
    prepare_severity_data, prepare_classification_data,
    train_linear_regression, train_random_forest_regressor, train_xgboost_regressor,
    evaluate_regression, train_random_forest_classifier, train_xgboost_classifier,
    evaluate_classification, plot_feature_importance, explain_with_shap,
    predict_premium, build_regression_comparison, build_classification_comparison
)

pd.set_option('display.max_columns', None)
print("Environment ready.")

## 1. Data Preparation

Load data, handle missing values, engineer features, and encode categoricals.

In [ ]:
DATA_PATH = r'c:\KAIM\MachineLearningRating_v3.txt'

try:
    df_raw = load_raw_data(DATA_PATH)
    df = preprocess(df_raw)
    
    # 1. Drop high-missing columns
    df = drop_high_missing(df, threshold=0.5)
    
    # 2. Engineer features
    df = engineer_features(df)
    
    # 3. Encode categoricals (Label encoding for efficiency in tree models)
    df_encoded = encode_categoricals(df, method='label')
    
    print(f"Final dataset shape: {df_encoded.shape}")
except FileNotFoundError:
    print("Data file not found.")

## 2. Claim Severity Prediction (Regression)

Predict `TotalClaims` for policies where `TotalClaims > 0`.

In [ ]:
X_train_sev, X_test_sev, y_train_sev, y_test_sev, feat_names_sev = prepare_severity_data(df_encoded)

results_sev = []

# 1. Linear Regression
lr_model = train_linear_regression(X_train_sev, y_train_sev)
results_sev.append(evaluate_regression(lr_model, X_test_sev, y_test_sev, "Linear Regression"))

# 2. Random Forest
rf_model = train_random_forest_regressor(X_train_sev, y_train_sev)
results_sev.append(evaluate_regression(rf_model, X_test_sev, y_test_sev, "Random Forest"))

# 3. XGBoost
xgb_model = train_xgboost_regressor(X_train_sev, y_train_sev)
results_sev.append(evaluate_regression(xgb_model, X_test_sev, y_test_sev, "XGBoost"))

comparison_sev = build_regression_comparison(results_sev)
comparison_sev

## 3. Claim Probability Prediction (Classification)

Predict `HasClaim` (0 or 1).

In [ ]:
X_train_clf, X_test_clf, y_train_clf, y_test_clf, feat_names_clf = prepare_classification_data(df_encoded)

results_clf = []

# 1. Random Forest Classifier
rf_clf = train_random_forest_classifier(X_train_clf, y_train_clf)
results_clf.append(evaluate_classification(rf_clf, X_test_clf, y_test_clf, "Random Forest"))

# 2. XGBoost Classifier
xgb_clf = train_xgboost_classifier(X_train_clf, y_train_clf)
results_clf.append(evaluate_classification(xgb_clf, X_test_clf, y_test_clf, "XGBoost"))

comparison_clf = build_classification_comparison(results_clf)
comparison_clf

## 4. Feature Importance & Interpretability

Using the best XGBoost model for insights.

In [ ]:
print("Top Features for Severity Model:")
plot_feature_importance(xgb_model, feat_names_sev, title="XGBoost Severity Importance")

print("Top Features for Claim Probability Model:")
plot_feature_importance(xgb_clf, feat_names_clf, title="XGBoost Probability Importance")

### SHAP Interpretation

In [ ]:
# This might take a few seconds
explain_with_shap(xgb_model, X_test_sev, feat_names_sev)

## 5. Premium Optimization

Calculate the recommended premium for the test set.

In [ ]:
X_full = df_encoded[feat_names_clf] # Assuming same features for simplicity
predicted_premiums = predict_premium(xgb_clf, xgb_model, X_full)

df_final = df.copy()
df_final['PredictedPremium'] = predicted_premiums

print("Comparison: Actual vs Predicted Premium")
df_final[['TotalPremium', 'PredictedPremium']].describe()